# 02 — MIMIC-IV-Note EDA

Interactive companion to `python -m src.eda.notes`. Works on the cached **per-note statistics** (`data/interim/eda_cache/note_stats.pkl`: lengths, token counts, section offsets, keyword flags — no text). Looking at actual note text is deliberately not part of this notebook; if you need to, do it locally and never copy it out.

> **Privacy rules (PhysioNet DUA) — read before running anything**
> 1. Never commit this notebook with outputs. Before `git add`: *Kernel → Restart & Clear Output*
>    (`tests/test_eda_privacy.py` fails if outputs are stored).
> 2. Don't print row-level data you'd paste anywhere (`df.head()`, note text, IDs). It's fine to look at it locally; it's not fine to copy it into a doc, chat, LLM prompt or slide.
> 3. Anything that leaves this notebook goes through `src.eda.privacy` (`safe_count`, `safe_rate`, `safe_quantiles`, …): counts < 11 are suppressed, no raw min/max.
> 4. Row-level caches live in `data/interim/eda_cache/` (gitignored). Never move them into `results/`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from src.config import load_config
from src.eda import loaders as L, notes as N
from src.eda.privacy import *

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
summary = json.loads((ROOT / 'results/eda/notes_summary.json').read_text())
summary['coverage']

In [ ]:
pd.DataFrame(summary['sections']['per_section'])

In [ ]:
pd.DataFrame(summary['keywords']['per_concept'])

## Per-note statistics (local only)

In [ ]:
cfg = load_config()
stats = pd.read_pickle(L.CACHE_DIR / 'note_stats.pkl')
stats = stats[stats['dup_rank'] == 0].merge(N._cohort(cfg), on='hadm_id', how='left')
TARGET = N.TARGET
stats.shape

In [ ]:
# Truncation x outcome x age band
stats['trunc_x_age'] = stats['age_band'] + ' / truncated=' + stats['truncated'].astype(str)
pd.DataFrame(rate_by_group(stats, 'trunc_x_age', TARGET))

In [ ]:
# Readmission rate by note length decile (is length itself a signal?)
stats['tok_decile'] = pd.qcut(stats['n_tokens'], 10, labels=False).astype(str)
pd.DataFrame(rate_by_group(stats, 'tok_decile', TARGET)).sort_values('category')

### Open questions worth chasing (see session 37)
- Would head+tail truncation (keep the discharge sections) help Stage 2?
- Followup Instructions are de-identified away — what does that mean for Stage 3's `no_followup_arranged` ground?
- Note length as a confounder: are long notes just sicker patients?